# 02 — Exploratory Data Analysis on Bronze

Distribution, correlation, and delay-pattern checks on the raw table. Nothing here
writes back — this notebook is documentation, not a pipeline step.

In [ ]:
import sys
sys.path.append("..")

from pyspark.sql.functions import avg, col, count, mean, stddev, when
from src import config

bronze = spark.table(config.BRONZE)
bronze_count = bronze.count()
print(f"Bronze rows: {bronze_count:,}")

## Column-level nulls

In [ ]:
null_report = bronze.select([
    (count(when(col(c).isNull(), c)) / bronze_count).alias(c)
    for c in bronze.columns
])
display(null_report)

## Delay distribution by year

In [ ]:
from pyspark.sql.functions import year, to_date

by_year = (
    bronze
    .withColumn("flight_year", year(to_date(col("FL_DATE"))))
    .groupBy("flight_year")
    .agg(
        count("*").alias("flights"),
        mean("ARR_DELAY").alias("mean_arr_delay"),
        stddev("ARR_DELAY").alias("stddev_arr_delay"),
    )
    .orderBy("flight_year")
)
display(by_year)

## Delay by airline

In [ ]:
by_airline = (
    bronze
    .groupBy("AIRLINE")
    .agg(count("*").alias("flights"), mean("ARR_DELAY").alias("mean_delay_min"))
    .orderBy(col("mean_delay_min").desc())
)
display(by_airline)

## Delay by hour of day
A monotonic afternoon rise in mean delay is the pattern most portfolios miss — capture
this chart as a screenshot for the dashboard grid.

In [ ]:
from pyspark.sql.functions import floor

by_hour = (
    bronze
    .withColumn("dep_hour", floor(col("CRS_DEP_TIME") / 100))
    .groupBy("dep_hour")
    .agg(count("*").alias("flights"), mean("ARR_DELAY").alias("mean_delay_min"))
    .orderBy("dep_hour")
)
display(by_hour)

## Notes for Silver
* 2020 has anomalously low delays (COVID) — Silver drops it.
* Actual times, taxi times, delay-cause breakdowns are dropped at Silver because
they are unavailable at booking time (target leakage for the pre-departure model).